In [1]:
import yfinance as yf

df = yf.download("SBK.JO", start="2018-01-01", end="2026-08-13")
df.columns = df.columns.get_level_values(0)  # flatten multi-level columns
df.to_csv("standardbank_jse.csv")
print(df.head())

[*********************100%***********************]  1 of 1 completed

Price              Close          High           Low          Open   Volume
Date                                                                       
2018-01-02  19330.451172  19627.949880  19005.093053  19402.089657  3146211
2018-01-03  18414.078125  19402.092367  18277.765989  19402.092367  2954003
2018-01-04  18664.810547  18685.705105  18115.582154  18341.442381  3910937
2018-01-05  18685.707031  18766.300336  18454.871886  18675.757240  1585832
2018-01-08  18878.732422  18956.340787  18394.177626  18705.606068  2655225


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("standardbank_jse.csv")
df.rename(columns={df.columns[0]: "Date"}, inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(df.head())
print(df.shape)
print("Missing values:\n", df.isna().sum())

        Date         Close          High           Low          Open   Volume
0 2018-01-02  19330.451172  19627.949880  19005.093053  19402.089657  3146211
1 2018-01-03  18414.078125  19402.092367  18277.765989  19402.092367  2954003
2 2018-01-04  18664.810547  18685.705105  18115.582154  18341.442381  3910937
3 2018-01-05  18685.707031  18766.300336  18454.871886  18675.757240  1585832
4 2018-01-08  18878.732422  18956.340787  18394.177626  18705.606068  2655225
(2152, 6)
Missing values:
 Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [3]:
df["Return"] = df["Close"].pct_change()

for lag in [1, 2, 3, 5]:
    df[f"Return_lag{lag}"] = df["Return"].shift(lag)

df["MA5"] = df["Close"].rolling(5).mean()
df["MA10"] = df["Close"].rolling(10).mean()
df["MA20"] = df["Close"].rolling(20).mean()

df["Close_to_MA5"] = df["Close"] / df["MA5"] - 1
df["Close_to_MA20"] = df["Close"] / df["MA20"] - 1

df["Volatility10"] = df["Return"].rolling(10).std()

delta = df["Close"].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
rs = gain / loss
df["RSI14"] = 100 - (100 / (1 + rs))

df["Volume_change"] = df["Volume"].pct_change()
df["Volume_MA5"] = df["Volume"].rolling(5).mean()

df["HL_spread"] = (df["High"] - df["Low"]) / df["Close"]

df.head()

,Date,Close,High,Low,Open,Volume,Return,Return_lag1,Return_lag2,Return_lag3,...,MA5,MA10,MA20,Close_to_MA5,Close_to_MA20,Volatility10,RSI14,Volume_change,Volume_MA5,HL_spread
0,2018-01-02,19330.451172,19627.949880,19005.093053,19402.089657,3146211,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032222
1,2018-01-03,18414.078125,19402.092367,18277.765989,19402.092367,2954003,-0.047406,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.061092,NaN,0.061058
2,2018-01-04,18664.810547,18685.705105,18115.582154,18341.442381,3910937,0.013616,-0.047406,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.323945,NaN,0.030545
3,2018-01-05,18685.707031,18766.300336,18454.871886,18675.757240,1585832,0.001120,0.013616,-0.047406,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.594514,NaN,0.016667
4,2018-01-08,18878.732422,18956.340787,18394.177626,18705.606068,2655225,0.010330,0.001120,0.013616,-0.047406,...,18794.755859,NaN,NaN,0.004468,NaN,NaN,NaN,0.674342,2850441.6,0.029778


In [4]:
df["Target_return"] = df["Close"].shift(-1) / df["Close"] - 1
df["Target_direction"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df_model = df.dropna().reset_index(drop=True)

feature_cols = [
    "Return_lag1", "Return_lag2", "Return_lag3", "Return_lag5",
    "Close_to_MA5", "Close_to_MA20", "Volatility10", "RSI14",
    "Volume_change", "Volume_MA5", "HL_spread"
]

split_date = "2025-01-01"
train = df_model[df_model["Date"] < split_date]
test = df_model[df_model["Date"] >= split_date]

X_train, y_train_ret = train[feature_cols], train["Target_return"]
X_test, y_test_ret = test[feature_cols], test["Target_return"]
y_train_dir, y_test_dir = train["Target_direction"], test["Target_direction"]

print(f"Train: {len(train)} rows, Test: {len(test)} rows")

Train: 1730 rows, Test: 402 rows


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

naive_pred = np.zeros(len(y_test_ret))
print(f"Naive baseline -> MAE: {mean_absolute_error(y_test_ret, naive_pred):.5f}, R2: 0.0000")

lr = LinearRegression()
lr.fit(X_train, y_train_ret)
lr_pred = lr.predict(X_test)
print(f"Linear Regression -> MAE: {mean_absolute_error(y_test_ret, lr_pred):.5f}, R2: {r2_score(y_test_ret, lr_pred):.4f}")

xgb_reg = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_reg.fit(X_train, y_train_ret)
xgb_pred = xgb_reg.predict(X_test)
print(f"XGBoost -> MAE: {mean_absolute_error(y_test_ret, xgb_pred):.5f}, R2: {r2_score(y_test_ret, xgb_pred):.4f}")

Naive baseline -> MAE: 0.26484, R2: 0.0000
Linear Regression -> MAE: 0.28400, R2: -0.0029
XGBoost -> MAE: 0.26531, R2: -0.0030


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

baseline_acc = max(y_test_dir.mean(), 1 - y_test_dir.mean())
print(f"Naive baseline -> Accuracy: {baseline_acc:.4f}")

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train_dir)
print(f"Logistic Regression -> Accuracy: {accuracy_score(y_test_dir, log_reg.predict(X_test)):.4f}")

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train_dir)
print(f"Random Forest -> Accuracy: {accuracy_score(y_test_dir, rf.predict(X_test)):.4f}")

xgb_clf = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
xgb_clf.fit(X_train, y_train_dir)
xgb_pred_dir = xgb_clf.predict(X_test)
print(f"XGBoost -> Accuracy: {accuracy_score(y_test_dir, xgb_pred_dir):.4f}")

print("\n", classification_report(y_test_dir, xgb_pred_dir, target_names=["Down", "Up"]))

Naive baseline -> Accuracy: 0.5423
Logistic Regression -> Accuracy: 0.4726
Random Forest -> Accuracy: 0.5597
XGBoost -> Accuracy: 0.5124

               precision    recall  f1-score   support

        Down       0.47      0.51      0.49       184
          Up       0.55      0.52      0.54       218

    accuracy                           0.51       402
   macro avg       0.51      0.51      0.51       402
weighted avg       0.52      0.51      0.51       402

